# Analysis for the Friedman benchmark

TODO Update with new SRB samples

The data sets are:

1. `f1a`: $f_1$ on 100 data points, no distractors
2. `f1b`: $f_1$ on 1000 data points, no distractors
3. `f1c`: $f_1$ on 100 data points, 5 columns of distractors
4. `f1d`: $f_1$ on 1000 data points, 5 columns of distractors

## Prelude

In [ ]:
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import sympy

import AnalysisUtils as au

## Loading data

In [ ]:
f1expr = sympy.sympify("10*sin(π*x1*x2) + 20*(x3 - 1/2)^2 + 10*x4 + 5*x5")
f1expr

In [ ]:
sympy.expand(f1expr)

These are the results of Jessamine symbolic regression.

In [ ]:
full_report = pd.read_csv("Generated/full-report.csv")

In [ ]:
full_report["sympy"] = full_report.expr_original_syms.apply(au.parse_if_needed)
full_report["sympy_defuzz"] = full_report.expr_original_syms_defuzz.apply(au.parse_if_needed)

In [ ]:
full_report.run_set.unique()

Make rows indexable by run set, data set, and sample number.

In [ ]:
full_report.sort_values(["run_set", "data_set", "mse"], inplace=True)
fr2 = full_report.set_index(["run_set", "data_set", "sample_num"])

In [ ]:
all_min_mse_ixs = fr2.groupby(level=["run_set", "data_set"]).mse.idxmin()
fr2.loc[all_min_mse_ixs]

In [ ]:
srb1_key = "SRB-2026-06-25-1715-arr8"
srb2_key = "SRB-2026-07-13-1130"
cht1_key = "CHT-2026-06-25-1700"
srb1 = fr2.loc[srb1_key]
srb2 = fr2.loc[srb2_key]
srb2s = fr2.loc[srb2_key + "-subset"]
cht1 = fr2.loc[cht1_key]
cht1s = fr2.loc[cht1_key + "-subset"]

In [ ]:
srb1.groupby(level=["data_set"]).size()

In [ ]:
srb2.groupby(level=["data_set"]).size()

In [ ]:
srb2s.groupby(level=["data_set"]).size()

In [ ]:
cht1.groupby(level=["data_set"]).size()

In [ ]:
cht1s.groupby(level=["data_set"]).size()

These are the best ones overall

In [ ]:
srb1_min_mse_ixs = srb1.groupby(level=["data_set"]).mse.idxmin()
srb2_min_mse_ixs = srb2.groupby(level=["data_set"]).mse.idxmin()
srb2s_min_mse_ixs = srb2s.groupby(level=["data_set"]).mse.idxmin()
cht1_min_mse_ixs = cht1.groupby(level=["data_set"]).mse.idxmin()
cht1s_min_mse_ixs = cht1s.groupby(level=["data_set"]).mse.idxmin()

In [ ]:
srb1.loc[srb1_min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
srb2.loc[srb2_min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
srb2s.loc[srb2s_min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
cht1.loc[cht1_min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
cht1s.loc[cht1s_min_mse_ixs, ["mse", "complexity_defuzz", "sympy_defuzz"]]

In [ ]:
au.count_by_threshold(srb1, threshold=1.0e-7)

### A bit about `srb1`

The best for `f1a` and `f1b` are exactly correct up to fuzz.
For `f1c`, it found several correct terms, but with one cruft term involving distractors $x_6$ and $x_7$, and a wrong term involving $x_3$ that seems to be an approximation for $20(x_3 - 1/2)^2$ using $1 - \cos (x_3 - 1/2) \approx (1/2)(x_3 - 1/2)^2$.
For `f1d`, it's just lost.

In [ ]:
(srb1.loc[srb1_min_mse_ixs]
 .sympy_defuzz
 .apply(lambda e: au.replace_near_integer(e, tolerance=1.0e-2)))

In [ ]:
(srb1.loc[srb1_min_mse_ixs]
 .sympy_defuzz
 .apply(lambda e: au.replace_near_integer(e, tolerance=1.0e-2)))

Looks like `srb2` is better.

Oddly, a lot of these have the correct terms, but they have a lot of extra cruft.
That means there's probably some numerical challenge with the process of solving for constants.

With generous defuzzing, we get 21 correct samples.

In [ ]:
srb1.loc["f1a"].sympy_defuzz.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=2.0e-2))

In [ ]:
srb1.loc["f1b"].sympy_defuzz.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=2.0e-2))

Not much luck here:

In [ ]:
srb1.loc["f1c"].sympy_defuzz.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=2.0e-2))

In [ ]:
srb1.loc["f1d"].sympy_defuzz.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=2.0e-2))

In [ ]:
sns.displot(data=srb1,
            x="complexity",
            y="mse",
            col="data_set",
            hue="data_set",
            log_scale=[False,True],
            )

## Main SRB and CHT results

In [ ]:
srb_key = srb2_key
srb = srb2
srb_min_mse_ixs = srb2_min_mse_ixs
srbs_key = srb2_key + "-subset"
srbs = srb2s
srbs_min_mse_ixs = srb2s_min_mse_ixs
cht_key = cht1_key
cht = cht1
cht_min_mse_ixs = cht1_min_mse_ixs
chts_key = cht1_key + "-subset"
chts = cht1s
chts_min_mse_ixs = cht1s_min_mse_ixs

In [ ]:
au.mse_threshold_table(fr2)

In [ ]:
srb_threshold_table = au.mse_threshold_table(fr2.loc[[srb_key]])
srb_threshold_table.to_csv("Generated/srb_threshold_table.csv")
srb_threshold_table.to_latex("Generated/srb_threshold_table.tex")
srb_threshold_table

In [ ]:
cht_threshold_table = au.mse_threshold_table(fr2.loc[[cht_key]])
cht_threshold_table.to_csv("Generated/cht_threshold_table.csv")
cht_threshold_table.to_latex("Generated/cht_threshold_table.tex")
cht_threshold_table

In [ ]:
plot_params = {
    "complexity_lims": (0, 449),
    "complexity_col": "complexity_defuzz",
    "mse_lims": (1.0e-12, 0.99e2),
    "complexity_binwidth": 20,
    "mse_binwidth": 0.8,
    "spiffy_titles": ["F1a", "F1b", "F1c", "F1d"],
    "xlabel": "Complexity (defuzzed)",
    "ylabel": "MSE",
}

In [ ]:
plot_params_bd = { **plot_params, "spiffy_titles": ["F1b", "F1d"] }

In [ ]:
fig = au.complexity_mse_displot(srb, file_stem="SRB-complexity-mse-displot", **plot_params)

In [ ]:
au.complexity_mse_displot(srbs, file_stem="srbs-complexity-mse-displot", **plot_params_bd)

In [ ]:
fig = au.complexity_mse_displot(cht, file_stem="cht-complexity-mse-displot", **plot_params)

In [ ]:
au.complexity_mse_displot(chts, file_stem="chts-complexity-mse-displot", **plot_params_bd)

In [ ]:
au.count_by_threshold(srb, threshold=1.0e-7)

In [ ]:
au.count_by_threshold(srbs, threshold=1.0e-7)

Looks like about 17 correct and a couple with cruft and some trig weirdness.

In [ ]:
srb.loc["f1a"].sympy_defuzz.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=2.0e-2))

Looks like 4 correct and 1 with trig weirdness.

In [ ]:
srb.loc["f1b"].sympy_defuzz.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=2.0e-2))

6 correct 2 with cruft.

In [ ]:
srb.loc["f1c"].sympy_defuzz.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=2.0e-2))

One almost correct but with cruft instead of the correct $x_5$ term

In [ ]:
srb.loc["f1d"].sympy_defuzz.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=2.0e-2))

The CHT config does a bit better.

In [ ]:
au.count_by_threshold(cht, threshold=1.0e-7)

In [ ]:
au.count_by_threshold(chts, threshold=1.0e-7)

In [ ]:
cht.loc["f1a"].sympy_defuzz.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=2.0e-2))

In [ ]:
cht.loc["f1b"].sympy_defuzz.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=2.0e-2))

In [ ]:
cht.loc["f1c"].sympy_defuzz.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=2.0e-2))

In [ ]:
cht.loc["f1d"].sympy_defuzz.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=2.0e-2))

## Does sub-setting help with the large data set?

It seems to make things just a little better, but probably not enough to write about.

In [ ]:
srbs.loc["f1b"].sympy_defuzz.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=2.0e-2))

In [ ]:
chts.loc["f1b"].sympy_defuzz.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=2.0e-2))

In [ ]:
srbs.loc["f1d"].sympy_defuzz.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=2.0e-2))

In [ ]:
chts.loc["f1d"].sympy_defuzz.apply(lambda e: au.replace_near_integer(sympy.expand(e), tolerance=2.0e-2))